# 🧠 Panopticon Protocol v3 — Production Training Notebook

> **Team:** Ayush Kumar & Ravi Prashant | **Target:** Meta PyTorch OpenEnv Grand Finale
>
> Fine-tunes **Qwen 2.5 1.5B** with LoRA using HuggingFace TRL's SFTTrainer.
> Curriculum chaining: easy → medium → hard → level_4 → level_5 (Manchurian).

### Fixes over previous version:
- ✅ Correct repo clone (panopticon-protocol-v3)
- ✅ Curriculum chains LoRA weights between levels
- ✅ Qwen chat template in training data
- ✅ Random seeds for trajectory diversity
- ✅ Evaluation step before Hub push
- ✅ Gradient checkpointing for T4 safety

⚠️ **REQUIREMENT**: `Runtime → Change runtime type → T4 GPU`

---
## 1️⃣ Install Dependencies

In [ ]:
!pip install -q torch trl transformers peft accelerate datasets gymnasium fastapi pydantic httpx tqdm numpy matplotlib

---
## 2️⃣ HuggingFace Authentication

Add your HF token as a Colab Secret named `HF_TOKEN` (click the key icon in sidebar), or enter it when prompted.

In [ ]:
import os
from huggingface_hub import login

# Use Colab Secrets (recommended) or manual input
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = input('Enter your HuggingFace token: ')

USERNAME = 'Ayush-Kumar0207'
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=True)
print(f'✅ Logged into HuggingFace as {USERNAME}')

---
## 3️⃣ Clone Repository & Verify

In [ ]:
# CRITICAL: Clone the CORRECT repository
!rm -rf /content/panopticon-protocol-v3
!git clone https://github.com/Ayush-Kumar0207/panopticon-protocol-v3.git /content/panopticon-protocol-v3
%cd /content/panopticon-protocol-v3

# Verify key files exist
import os
required = ['environment.py', 'models.py', 'grader.py', 'train_trl_v2.py']
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status} {f}')

---
## 4️⃣ Smoke Test — Verify Environment Works

In [ ]:
%cd /content/panopticon-protocol-v3
!python smoke_test.py

---
## 5️⃣ Verify GPU Available

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory
    print(f'VRAM: {mem / 1e9:.1f} GB')
else:
    print('❌ No GPU! Go to Runtime > Change runtime type > T4 GPU')

---
## 6️⃣ Full Curriculum Training (Easy → Manchurian)

Chains LoRA adapters across difficulty levels with proper chat template.

**Estimated time on T4:**
- `--episodes 30` ≈ 1.5h (fast)
- `--episodes 50` ≈ 2.5h (recommended)
- `--episodes 100` ≈ 4h (best quality)

In [ ]:
%cd /content/panopticon-protocol-v3

!python train_trl_v2.py \
    --curriculum \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 50

---
## 7️⃣ Evaluate Trained Model

Tests the final Level 5 model on **unseen** game seeds.

In [ ]:
%cd /content/panopticon-protocol-v3

from train_trl_v2 import evaluate_model

avg_score = evaluate_model('trl_model_level_5', task_level='level_5', num_games=5)
print(f'\n{"="*40}')
print(f'  FINAL AVERAGE GRADE: {avg_score:.3f}')
print(f'{"="*40}')

---
## 8️⃣ Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = f'{USERNAME}/panopticon-argus-qwen-1.5B'

try:
    api.create_repo(repo_id, exist_ok=True)
    print(f'Created/Verified repository: {repo_id}')

    if os.path.exists('trl_model_level_5'):
        print('Pushing model to HuggingFace Hub...')
        api.upload_folder(
            folder_path='trl_model_level_5',
            repo_id=repo_id,
            commit_message='Upload Panopticon ARGUS agent (curriculum-trained, Level 5)'
        )
        print(f'✅ Pushed to https://huggingface.co/{repo_id}')
    else:
        print('⚠️ trl_model_level_5 not found. Check training output above.')
except Exception as e:
    print(f'❌ Push failed: {e}')

---
## 9️⃣ (Optional) Quick Single-Level Test

Fast test on one level (~20 min) before committing to full curriculum:

In [ ]:
# Uncomment to run:
# %cd /content/panopticon-protocol-v3
# !python train_trl_v2.py --level easy --model Qwen/Qwen2.5-1.5B-Instruct --episodes 20 --eval --eval-games 3

---
## 📎 Links

- **HF Space**: [panopticon-protocol-v3](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-protocol-v3)
- **GitHub**: [panopticon-protocol-v3](https://github.com/Ayush-Kumar0207/panopticon-protocol-v3)

*Meta PyTorch OpenEnv Hackathon x Scaler Grand Finale, April 2026*

*Team: Ayush Kumar & Ravi Prashant*